In [ ]:
import os

luna_root = '/kaggle/input/datasets/vafaeii/luna16'

for root, dirs, files in os.walk(luna_root):
    depth = root.replace(luna_root, '').count(os.sep)
    if depth <= 2:
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/")
        if depth == 2:
            fnames = files[:5]
            for f in fnames:
                print(f"{'  ' * (depth+1)}{f}")
            if len(files) > 5:
                print(f"{'  ' * (depth+1)}... ({len(files)} fichiers total)")

In [ ]:
import os
import glob

luna_root = '/kaggle/input/datasets/vafaeii/luna16'

# Chercher tous les CSV
csv_files = glob.glob(os.path.join(luna_root, '**/*.csv'), recursive=True)
for f in csv_files:
    print(f)

In [ ]:
import pandas as pd

luna_root = '/kaggle/input/datasets/vafaeii/luna16'

print("=== annotations.csv ===")
ann = pd.read_csv(f'{luna_root}/annotations.csv')
print(ann.shape)
print(ann.head())
print(ann.columns.tolist())

print("\n=== candidates_V2.csv ===")
cand = pd.read_csv(f'{luna_root}/candidates_V2/candidates_V2.csv')
print(cand.shape)
print(cand.head())
print(cand['class'].value_counts())

In [ ]:
import SimpleITK as sitk
import numpy as np
import glob

luna_root = '/kaggle/input/datasets/vafaeii/luna16'

# Prendre le premier .mhd de subset0
mhd_files = glob.glob(f'{luna_root}/subset0/subset0/*.mhd')
sample_path = mhd_files[0]
print(f"Fichier : {sample_path}")

ct = sitk.ReadImage(sample_path)
arr = sitk.GetArrayFromImage(ct)  # shape: (Z, Y, X)

print(f"\nShape array     : {arr.shape}")
print(f"Spacing (mm)    : {ct.GetSpacing()}")
print(f"Origin          : {ct.GetOrigin()}")
print(f"Direction       : {ct.GetDirection()}")
print(f"dtype           : {arr.dtype}")
print(f"Min / Max HU    : {arr.min()} / {arr.max()}")
print(f"Mean / Std HU   : {arr.mean():.1f} / {arr.std():.1f}")

In [ ]:
import os
import glob
import pandas as pd

luna_root = '/kaggle/input/datasets/vafaeii/luna16'

# Compter les scans par subset
total_mhd = 0
for i in range(10):
    mhd_files = glob.glob(f'{luna_root}/subset{i}/subset{i}/*.mhd')
    print(f"subset{i} : {len(mhd_files)} scans")
    total_mhd += len(mhd_files)
print(f"\nTotal scans : {total_mhd}")

# Distribution nodules par scan
ann = pd.read_csv(f'{luna_root}/annotations.csv')
nodules_per_scan = ann.groupby('seriesuid').size()
print(f"\nScans avec nodules    : {len(nodules_per_scan)}")
print(f"Nodules par scan      : min={nodules_per_scan.min()}, max={nodules_per_scan.max()}, mean={nodules_per_scan.mean():.2f}")
print(f"\nDistribution diameter_mm :")
print(ann['diameter_mm'].describe().round(2))

# Scans sans nodules
all_uids = set()
for i in range(10):
    for f in glob.glob(f'{luna_root}/subset{i}/subset{i}/*.mhd'):
        all_uids.add(os.path.basename(f).replace('.mhd', ''))
annotated_uids = set(ann['seriesuid'].unique())
print(f"\nScans sans nodule     : {len(all_uids - annotated_uids)}")
print(f"Scans avec nodule     : {len(all_uids & annotated_uids)}")

In [ ]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
import glob

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
ann = pd.read_csv(f'{luna_root}/annotations.csv')

# Prendre un scan qui a des nodules
sample_uid = ann['seriesuid'].iloc[0]
print(f"UID : {sample_uid}")

# Trouver le fichier
mhd_path = None
for i in range(10):
    candidates = glob.glob(f'{luna_root}/subset{i}/subset{i}/{sample_uid}.mhd')
    if candidates:
        mhd_path = candidates[0]
        break
print(f"Fichier : {mhd_path}")

ct = sitk.ReadImage(mhd_path)
arr = sitk.GetArrayFromImage(ct)  # (Z, Y, X)

origin  = np.array(ct.GetOrigin())   # (x, y, z)
spacing = np.array(ct.GetSpacing())  # (x, y, z)

# Nodules de ce scan
nodules = ann[ann['seriesuid'] == sample_uid]
print(f"\n{len(nodules)} nodule(s) dans ce scan :")
print(nodules[['coordX','coordY','coordZ','diameter_mm']].to_string())

# Conversion world → voxel
print("\nCoordonnées voxel (z, y, x) :")
for _, row in nodules.iterrows():
    world = np.array([row['coordX'], row['coordY'], row['coordZ']])
    voxel = (world - origin) / spacing  # (vx, vy, vz)
    vz, vy, vx = voxel[2], voxel[1], voxel[0]
    print(f"  voxel=({vz:.1f}, {vy:.1f}, {vx:.1f})  |  scan shape={arr.shape}  |  diam={row['diameter_mm']:.1f}mm")
    # Vérifier que c'est dans les bornes
    in_bounds = (0<=vz<arr.shape[0]) and (0<=vy<arr.shape[1]) and (0<=vx<arr.shape[2])
    print(f"  in_bounds={in_bounds}")

In [ ]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
ann = pd.read_csv(f'{luna_root}/annotations.csv')

sample_uid = ann['seriesuid'].iloc[0]
mhd_path = glob.glob(f'{luna_root}/subset8/subset8/{sample_uid}.mhd')[0]

ct = sitk.ReadImage(mhd_path)
arr = sitk.GetArrayFromImage(ct)  # (Z, Y, X)
origin  = np.array(ct.GetOrigin())
spacing = np.array(ct.GetSpacing())

nodules = ann[ann['seriesuid'] == sample_uid]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for nod_i, (_, row) in enumerate(nodules.iterrows()):
    world = np.array([row['coordX'], row['coordY'], row['coordZ']])
    voxel = (world - origin) / spacing
    vz, vy, vx = int(round(voxel[2])), int(round(voxel[1])), int(round(voxel[0]))
    diam_pix = row['diameter_mm'] / spacing[0]  # en pixels

    # 3 coupes : axiale, coronale, sagittale
    slices = [
        arr[vz, :, :],           # axiale
        arr[:, vy, :],           # coronale
        arr[:, :, vx],           # sagittale
    ]
    titles = ['Axiale', 'Coronale', 'Sagittale']
    centers = [(vy, vx), (vz, vx), (vz, vy)]

    for j, (sl, title, (cy, cx)) in enumerate(zip(slices, titles, centers)):
        ax = axes[nod_i][j]
        # Crop 64px autour du nodule
        s = 64
        y0, y1 = max(0, cy-s), min(sl.shape[0], cy+s)
        x0, x1 = max(0, cx-s), min(sl.shape[1], cx+s)
        crop = sl[y0:y1, x0:x1]
        # Fenêtrage pulmonaire [-1000, 400]
        crop_win = np.clip(crop, -1000, 400)
        ax.imshow(crop_win, cmap='gray')
        # Cercle nodule
        circle = plt.Circle((cx-x0, cy-y0), diam_pix/2, color='red', fill=False, lw=1.5)
        ax.add_patch(circle)
        ax.set_title(f'Nod{nod_i+1} {title} | diam={row["diameter_mm"]:.1f}mm')
        ax.axis('off')

plt.tight_layout()
plt.savefig('nodule_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print("Sauvegardé.")

In [ ]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
import glob

luna_root = '/kaggle/input/datasets/vafaeii/luna16'

# Ouvrir 5 scans variés et regarder leurs dimensions
sample_paths = []
for i in [0, 3, 5, 7, 9]:
    files = glob.glob(f'{luna_root}/subset{i}/subset{i}/*.mhd')
    sample_paths.append(files[0])

print(f"{'Scan':<10} {'Shape (Z,Y,X)':<20} {'Spacing X':<12} {'Spacing Z':<12}")
print("-" * 55)

shapes = []
for p in sample_paths:
    ct = sitk.ReadImage(p)
    arr = sitk.GetArrayFromImage(ct)
    sp = ct.GetSpacing()
    shapes.append(arr.shape)
    name = os.path.basename(p)[:20]
    print(f"{name:<10} {str(arr.shape):<20} {sp[0]:.4f}mm     {sp[2]:.4f}mm")

print("\n--- Simulation extraction patches 64x64 ---")
# Pour un scan typique 512x512x200, patch_size=64, stride=64
patch_size = 64
for shape in shapes:
    Z, Y, X = shape
    n_x = X // patch_size
    n_y = Y // patch_size
    n_z = Z
    total = n_x * n_y * n_z
    print(f"Shape {shape} → {n_x}×{n_y} patches/tranche × {n_z} tranches = {total:,} patches bruts")

In [ ]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
import glob

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
ann = pd.read_csv(f'{luna_root}/annotations.csv')

# Simuler avec masque poumon (Otsu) comme dans CAMELYON
# Sur un scan typique, combien de patches contiennent du tissu pulmonaire ?

mhd_path = glob.glob(f'{luna_root}/subset0/subset0/*.mhd')[0]
ct = sitk.ReadImage(mhd_path)
arr = sitk.GetArrayFromImage(ct)  # (Z, Y, X), int16, HU

patch_size = 64
fg_thresh = 0.01  # 1% de pixels foreground minimum (comme CAMELYON)

# Fenêtrage pulmonaire : tissu = HU > -800 (air=-1000, poumon=-800 à -400)
fg_mask = arr > -800  # True = tissu (pas air)

total_patches = 0
fg_patches = 0
nodule_patches = 0

Z, Y, X = arr.shape

# Nodules de ce scan
uid = os.path.basename(mhd_path).replace('.mhd','')
origin  = np.array(ct.GetOrigin())
spacing = np.array(ct.GetSpacing())
scan_nodules = ann[ann['seriesuid'] == uid]

for z in range(Z):
    for y in range(0, Y - patch_size + 1, patch_size):
        for x in range(0, X - patch_size + 1, patch_size):
            total_patches += 1
            patch_fg = fg_mask[z, y:y+patch_size, x:x+patch_size]
            if patch_fg.mean() >= fg_thresh:
                fg_patches += 1
                # Ce patch contient-il un nodule ?
                for _, row in scan_nodules.iterrows():
                    world = np.array([row['coordX'], row['coordY'], row['coordZ']])
                    voxel = (world - origin) / spacing
                    vz, vy, vx = voxel[2], voxel[1], voxel[0]
                    if (z == int(round(vz)) and
                        y <= vy < y+patch_size and
                        x <= vx < x+patch_size):
                        nodule_patches += 1

print(f"Shape scan         : {arr.shape}")
print(f"Total patches brut : {total_patches:,}")
print(f"Patches foreground : {fg_patches:,}  ({100*fg_patches/total_patches:.1f}%)")
print(f"Patches avec nodule: {nodule_patches}")
print(f"\n→ Ratio fg/total   : {fg_patches/total_patches:.2f}")
print(f"→ Patches fg × 888 scans ≈ {fg_patches * 888:,} patches total estimé")

In [ ]:
import SimpleITK as sitk
import numpy as np
from skimage.filters import threshold_otsu
import glob

luna_root = '/kaggle/input/datasets/vafaeii/luna16'

mhd_path = glob.glob(f'{luna_root}/subset0/subset0/*.mhd')[0]
ct = sitk.ReadImage(mhd_path)
arr = sitk.GetArrayFromImage(ct)

patch_size = 64
fg_thresh = 0.01

# Tester 3 stratégies de masque foreground
strategies = {
    'HU > -800 (tissu brut)'      : arr > -800,
    'HU > -600 (tissu dense)'     : arr > -600,
    'HU in [-800, 400] (poumon+tissu)' : (arr > -800) & (arr < 400),
}

# + Otsu sur les valeurs clipées
arr_clip = np.clip(arr, -1000, 400).astype(np.float32)
otsu_val = threshold_otsu(arr_clip)
strategies[f'Otsu (seuil={otsu_val:.0f})'] = arr_clip > otsu_val

print(f"Shape : {arr.shape}  |  patch_size={patch_size}  |  fg_thresh={fg_thresh}\n")
print(f"{'Stratégie':<40} {'Patches FG':>12} {'% total':>8}")
print("-" * 62)

Z, Y, X = arr.shape
total = (Z) * (Y // patch_size) * (X // patch_size)

for name, mask in strategies.items():
    fg_count = 0
    for z in range(Z):
        for y in range(0, Y - patch_size + 1, patch_size):
            for x in range(0, X - patch_size + 1, patch_size):
                if mask[z, y:y+patch_size, x:x+patch_size].mean() >= fg_thresh:
                    fg_count += 1
    print(f"{name:<40} {fg_count:>12,} {100*fg_count/total:>7.1f}%")

print(f"\nTotal patches brut : {total:,}")
print(f"Estimation 888 scans (×0.75 approx) :")
for name, _ in strategies.items():
    pass  # juste pour spacing
print(f"  ~{int(total * 0.4 * 888):,} à ~{int(total * 0.73 * 888):,} patches selon stratégie")

In [ ]:
import SimpleITK as sitk
import numpy as np
import glob
import os

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
seg_dir = f'{luna_root}/seg-lungs-LUNA16/seg-lungs-LUNA16'

# Regarder le format des fichiers de segmentation
seg_mhd = glob.glob(f'{seg_dir}/*.mhd')
print(f"Nombre de masques de segmentation : {len(seg_mhd)}")

# Prendre un scan qui a un masque
sample_uid = os.path.basename(seg_mhd[0]).replace('.mhd','')
print(f"\nUID : {sample_uid}")

# Charger le CT correspondant
ct_path = None
for i in range(10):
    c = glob.glob(f'{luna_root}/subset{i}/subset{i}/{sample_uid}.mhd')
    if c:
        ct_path = c[0]
        break
print(f"CT trouvé : {ct_path is not None}")

# Charger CT + masque
ct  = sitk.ReadImage(ct_path)
seg = sitk.ReadImage(seg_mhd[0])

arr_ct  = sitk.GetArrayFromImage(ct)
arr_seg = sitk.GetArrayFromImage(seg)

print(f"\nShape CT  : {arr_ct.shape}")
print(f"Shape SEG : {arr_seg.shape}")
print(f"Valeurs uniques SEG : {np.unique(arr_seg)}")
print(f"% pixels poumon : {(arr_seg > 0).mean()*100:.1f}%")

# Compter patches foreground avec masque seg
patch_size = 64
fg_thresh  = 0.01
Z, Y, X = arr_ct.shape
fg_count = 0
total = 0
for z in range(Z):
    for y in range(0, Y - patch_size + 1, patch_size):
        for x in range(0, X - patch_size + 1, patch_size):
            total += 1
            if arr_seg[z, y:y+patch_size, x:x+patch_size].mean() >= fg_thresh:
                fg_count += 1

print(f"\nAvec masque seg officiel :")
print(f"Patches FG : {fg_count:,} / {total:,}  ({100*fg_count/total:.1f}%)")
print(f"Estimation 888 scans : ~{int(fg_count/total * total * 888):,} patches")

In [ ]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
import glob
import os

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
seg_dir   = f'{luna_root}/seg-lungs-LUNA16/seg-lungs-LUNA16'
ann = pd.read_csv(f'{luna_root}/annotations.csv')

# Tester sur 20 scans annotés : est-ce que les nodules tombent dans le masque seg ?
seg_mhd_files = {os.path.basename(f).replace('.mhd',''): f 
                 for f in glob.glob(f'{seg_dir}/*.mhd')}

# Scans annotés qui ont un masque seg
ann_uids = ann['seriesuid'].unique()
common = [uid for uid in ann_uids if uid in seg_mhd_files]
print(f"Scans annotés avec masque seg : {len(common)} / {len(ann_uids)}")

covered = 0
not_covered = 0
total_nodules = 0

for uid in common[:20]:  # tester sur 20
    # Charger seg
    seg = sitk.ReadImage(seg_mhd_files[uid])
    arr_seg = sitk.GetArrayFromImage(seg)
    
    # Charger CT pour origin/spacing
    ct_path = None
    for i in range(10):
        c = glob.glob(f'{luna_root}/subset{i}/subset{i}/{uid}.mhd')
        if c:
            ct_path = c[0]; break
    ct = sitk.ReadImage(ct_path)
    origin  = np.array(ct.GetOrigin())
    spacing = np.array(ct.GetSpacing())
    
    # Vérifier chaque nodule
    for _, row in ann[ann['seriesuid']==uid].iterrows():
        total_nodules += 1
        world = np.array([row['coordX'], row['coordY'], row['coordZ']])
        voxel = (world - origin) / spacing
        vz = int(round(voxel[2]))
        vy = int(round(voxel[1]))
        vx = int(round(voxel[0]))
        
        # Clamp aux bornes
        vz = np.clip(vz, 0, arr_seg.shape[0]-1)
        vy = np.clip(vy, 0, arr_seg.shape[1]-1)
        vx = np.clip(vx, 0, arr_seg.shape[2]-1)
        
        val = arr_seg[vz, vy, vx]
        if val > 0:
            covered += 1
        else:
            not_covered += 1
            print(f"  NON couvert : uid={uid[:30]}... voxel=({vz},{vy},{vx}) seg_val={val}")

print(f"\nNodules testés  : {total_nodules}")
print(f"Couverts par seg : {covered} ({100*covered/total_nodules:.1f}%)")
print(f"Non couverts    : {not_covered} ({100*not_covered/total_nodules:.1f}%)")

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
ann  = pd.read_csv(f'{luna_root}/annotations.csv')
cand = pd.read_csv(f'{luna_root}/candidates_V2/candidates_V2.csv')

# ── Tâche : classification binaire par SCAN ──
# Comme CAMELYON (lame positive/négative), ici scan positif/négatif
all_uids = []
for i in range(10):
    for f in glob.glob(f'{luna_root}/subset{i}/subset{i}/*.mhd'):
        uid = os.path.basename(f).replace('.mhd','')
        subset = i
        all_uids.append({'seriesuid': uid, 'subset': subset})

df_scans = pd.DataFrame(all_uids)
ann_uids = set(ann['seriesuid'].unique())
df_scans['label'] = df_scans['seriesuid'].apply(lambda x: 1 if x in ann_uids else 0)

print("=== Tâche : détection de nodule (classification scan) ===")
print(f"Total scans   : {len(df_scans)}")
print(f"Positifs (1)  : {df_scans['label'].sum()}")
print(f"Négatifs (0)  : {(df_scans['label']==0).sum()}")
print(f"Ratio pos/neg : {df_scans['label'].mean():.2f}")

print("\n=== Distribution par subset ===")
for i in range(10):
    sub = df_scans[df_scans['subset']==i]
    pos = sub['label'].sum()
    neg = len(sub) - pos
    print(f"  subset{i} : {len(sub)} scans  |  pos={pos}  neg={neg}")

print("\n=== Résumé pipeline (comme CAMELYON) ===")
print("1. Extraction patches 64×64 px sur tranches 2D")
print("   → filtre masque seg officiel (≥1% pixels poumon)")
print("   → HU clippé [-1000, 400] + normalisé")
print("   → ~4.2M patches estimés")
print("2. Pré-entraînement BYOL sur ces patches (ResNet18, grayscale 1 canal)")
print("3. Extraction features → HDF5 (1 groupe par scan)")
print("4. Classification avec ton modèle IPS+attention")

In [ ]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
import glob
import os

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
seg_dir   = f'{luna_root}/seg-lungs-LUNA16/seg-lungs-LUNA16'

# Construire le mapping uid → (subset, ct_path, seg_path)
seg_files = {os.path.basename(f).replace('.mhd',''): f 
             for f in glob.glob(f'{seg_dir}/*.mhd')}

uid_to_path = {}
for i in range(10):
    for f in glob.glob(f'{luna_root}/subset{i}/subset{i}/*.mhd'):
        uid = os.path.basename(f).replace('.mhd','')
        uid_to_path[uid] = {'ct': f, 'subset': i,
                            'seg': seg_files.get(uid, None)}

print(f"Total scans indexés     : {len(uid_to_path)}")
print(f"Scans avec masque seg   : {sum(1 for v in uid_to_path.values() if v['seg'])}")
print(f"Scans sans masque seg   : {sum(1 for v in uid_to_path.values() if not v['seg'])}")

# Tester l'extraction sur 1 scan complet et compter les patches
sample_uid = list(uid_to_path.keys())[0]
info = uid_to_path[sample_uid]

ct  = sitk.ReadImage(info['ct'])
seg = sitk.ReadImage(info['seg'])
arr_ct  = sitk.GetArrayFromImage(ct).astype(np.float32)
arr_seg = sitk.GetArrayFromImage(seg)

# Prétraitement
patch_size = 64
fg_thresh  = 0.01
arr_clip = np.clip(arr_ct, -1000, 400)
arr_norm = (arr_clip - (-1000)) / (400 - (-1000))  # → [0, 1]

Z, Y, X = arr_ct.shape
patches_kept = []

for z in range(Z):
    for y in range(0, Y - patch_size + 1, patch_size):
        for x in range(0, X - patch_size + 1, patch_size):
            seg_patch = arr_seg[z, y:y+patch_size, x:x+patch_size]
            if seg_patch.mean() >= fg_thresh:
                patch = arr_norm[z, y:y+patch_size, x:x+patch_size]
                patches_kept.append((z, y, x))

print(f"\nScan : {sample_uid[:40]}...")
print(f"Shape         : {arr_ct.shape}")
print(f"Patches gardés: {len(patches_kept):,}")
print(f"Ex. coords    : {patches_kept[:5]}")
print(f"\nPatch valeurs min/max : {arr_norm[patches_kept[0][0], patches_kept[0][1]:patches_kept[0][1]+patch_size, patches_kept[0][2]:patches_kept[0][2]+patch_size].min():.3f} / {arr_norm[patches_kept[0][0], patches_kept[0][1]:patches_kept[0][1]+patch_size, patches_kept[0][2]:patches_kept[0][2]+patch_size].max():.3f}")

In [ ]:
import SimpleITK as sitk
import numpy as np
import glob
import os
import time

luna_root = '/kaggle/input/datasets/vafaeii/luna16'
seg_dir   = f'{luna_root}/seg-lungs-LUNA16/seg-lungs-LUNA16'

seg_files = {os.path.basename(f).replace('.mhd',''): f 
             for f in glob.glob(f'{seg_dir}/*.mhd')}

uid_to_path = {}
for i in range(10):
    for f in glob.glob(f'{luna_root}/subset{i}/subset{i}/*.mhd'):
        uid = os.path.basename(f).replace('.mhd','')
        uid_to_path[uid] = {'ct': f, 'subset': i, 'seg': seg_files.get(uid)}

# Mesurer sur 10 scans pour estimer temps total + nombre de patches
patch_size = 64
fg_thresh  = 0.01
uids_sample = list(uid_to_path.keys())[:10]

total_patches_sample = 0
t0 = time.time()

for uid in uids_sample:
    info = uid_to_path[uid]
    ct  = sitk.ReadImage(info['ct'])
    seg = sitk.ReadImage(info['seg'])
    arr_ct  = sitk.GetArrayFromImage(ct).astype(np.float32)
    arr_seg = sitk.GetArrayFromImage(seg)

    arr_norm = (np.clip(arr_ct, -1000, 400) + 1000) / 1400.0

    Z, Y, X = arr_ct.shape
    count = 0
    for z in range(Z):
        seg_slice = arr_seg[z]
        for y in range(0, Y - patch_size + 1, patch_size):
            for x in range(0, X - patch_size + 1, patch_size):
                if seg_slice[y:y+patch_size, x:x+patch_size].mean() >= fg_thresh:
                    count += 1
    total_patches_sample += count

elapsed = time.time() - t0
per_scan = elapsed / 10

print(f"10 scans traités en   : {elapsed:.1f}s  ({per_scan:.1f}s/scan)")
print(f"Patches sur 10 scans  : {total_patches_sample:,}  (~{total_patches_sample//10:,}/scan)")
print(f"\nEstimation 888 scans  :")
print(f"  Patches total       : ~{total_patches_sample//10 * 888:,}")
print(f"  Temps extraction    : ~{per_scan * 888 / 60:.0f} minutes")
print(f"\nComparaison CAMELYON  : 28.6M patches, ~71800/lame")
print(f"LUNA16 estimé         : {total_patches_sample//10 * 888:,} patches, ~{total_patches_sample//10:,}/scan")

# On commence par luna_utils.py :

In [ ]:
# data/luna16/luna_utils.py

import os
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk
from collections import OrderedDict


LUNA_ROOT = '/kaggle/input/datasets/vafaeii/luna16'
SEG_DIR   = os.path.join(LUNA_ROOT, 'seg-lungs-LUNA16', 'seg-lungs-LUNA16')


def build_uid_index(luna_root=LUNA_ROOT, seg_dir=SEG_DIR):
    """
    Construit un dictionnaire uid → {ct, seg, subset, label}
    pour les 888 scans LUNA16.
    """
    ann = pd.read_csv(os.path.join(luna_root, 'annotations.csv'))
    ann_uids = set(ann['seriesuid'].unique())

    seg_files = {
        os.path.basename(f).replace('.mhd', ''): f
        for f in glob.glob(os.path.join(seg_dir, '*.mhd'))
    }

    index = OrderedDict()
    for i in range(10):
        pattern = os.path.join(luna_root, f'subset{i}', f'subset{i}', '*.mhd')
        for ct_path in sorted(glob.glob(pattern)):
            uid = os.path.basename(ct_path).replace('.mhd', '')
            index[uid] = {
                'ct'    : ct_path,
                'seg'   : seg_files.get(uid),
                'subset': i,
                'label' : 1 if uid in ann_uids else 0,
            }
    return index


def world_to_voxel(coord_world, origin, spacing):
    """
    Convertit coordonnées monde (mm) → voxel (x, y, z).

    Parameters
    ----------
    coord_world : array-like (3,)  [x, y, z] en mm
    origin      : array-like (3,)  origine du CT en mm
    spacing     : array-like (3,)  taille voxel en mm

    Returns
    -------
    np.ndarray (3,) [vx, vy, vz] en pixels (float)
    """
    return (np.array(coord_world) - np.array(origin)) / np.array(spacing)


def load_ct_seg(info):
    """
    Charge un CT + son masque seg, retourne arrays normalisés.

    Returns
    -------
    arr_norm : np.ndarray float32 (Z, Y, X)  valeurs dans [0, 1]
    arr_seg  : np.ndarray uint8   (Z, Y, X)  valeurs 0/3/4/5
    origin   : np.ndarray (3,)
    spacing  : np.ndarray (3,)
    """
    ct  = sitk.ReadImage(info['ct'])
    seg = sitk.ReadImage(info['seg'])

    arr_ct  = sitk.GetArrayFromImage(ct).astype(np.float32)
    arr_seg = sitk.GetArrayFromImage(seg).astype(np.uint8)

    # Fenêtrage pulmonaire + normalisation min-max → [0, 1]
    arr_norm = (np.clip(arr_ct, -1000, 400) + 1000) / 1400.0

    origin  = np.array(ct.GetOrigin())
    spacing = np.array(ct.GetSpacing())

    return arr_norm, arr_seg, origin, spacing


def get_foreground_patches(arr_norm, arr_seg,
                           patch_size=64, fg_thresh=0.01):
    """
    Itère sur toutes les tranches 2D et retourne les coordonnées
    (z, y, x) des patches dont le masque seg ≥ fg_thresh.

    Yields
    ------
    (z, y, x) : int, int, int
    """
    Z, Y, X = arr_norm.shape
    for z in range(Z):
        seg_slice = arr_seg[z]
        for y in range(0, Y - patch_size + 1, patch_size):
            for x in range(0, X - patch_size + 1, patch_size):
                if seg_slice[y:y+patch_size, x:x+patch_size].mean() >= fg_thresh:
                    yield z, y, x

# foreground

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import multiprocessing as mp

# Pas d'import — build_uid_index, load_ct_seg, get_foreground_patches
# sont déjà définis dans la cellule précédente

LUNA_ROOT  = '/kaggle/input/datasets/vafaeii/luna16'
SEG_DIR    = f'{LUNA_ROOT}/seg-lungs-LUNA16/seg-lungs-LUNA16'
OUT_DIR    = '/kaggle/working/luna16_coords'
PATCH_SIZE = 64
FG_THRESH  = 0.01
TEST_FOLD  = 0
N_WORKER   = 4

UID_INDEX = build_uid_index(LUNA_ROOT, SEG_DIR)

def get_foreground_coords(uid):
    info = UID_INDEX[uid]
    try:
        arr_norm, arr_seg, origin, spacing = load_ct_seg(info)
        x_vals, y_vals, z_vals = [], [], []
        for z, y, x in get_foreground_patches(arr_norm, arr_seg,
                                               patch_size=PATCH_SIZE,
                                               fg_thresh=FG_THRESH):
            x_vals.append(x)
            y_vals.append(y)
            z_vals.append(z)
        return uid, x_vals, y_vals, z_vals
    except Exception as e:
        print(f"ERR {uid[:40]} : {e}")
        return uid, [], [], []

def build_coords_bounds(uids, subset_name):
    pool = mp.Pool(N_WORKER)
    results = list(tqdm(pool.imap(get_foreground_coords, uids), total=len(uids)))
    pool.close()
    pool.join()

    all_names, all_x, all_y, all_z = [], [], [], []
    all_pos_id, all_glob_id = [], []
    start_ids, end_ids, bound_names = [], [], []

    global_id = 0
    for uid, x_vals, y_vals, z_vals in results:
        if len(x_vals) == 0:
            continue
        n = len(x_vals)
        start_ids.append(global_id)
        end_ids.append(global_id + n - 1)
        bound_names.append(uid)

        for pos_id, (x, y, z) in enumerate(zip(x_vals, y_vals, z_vals)):
            all_names.append(uid)
            all_x.append(x)
            all_y.append(y)
            all_z.append(z)
            all_pos_id.append(pos_id)
            all_glob_id.append(global_id)
            global_id += 1

    coords_df = pd.DataFrame({
        'id'    : all_glob_id,
        'pos_id': all_pos_id,
        'name'  : all_names,
        'x'     : all_x,
        'y'     : all_y,
        'z'     : all_z,
    })
    bounds_df = pd.DataFrame({
        'names'   : bound_names,
        'start_id': start_ids,
        'end_id'  : end_ids,
    })

    os.makedirs(OUT_DIR, exist_ok=True)
    with open(f'{OUT_DIR}/coords_{subset_name}.pkl', 'wb') as f:
        pickle.dump(coords_df, f)
    with open(f'{OUT_DIR}/bounds_{subset_name}.pkl', 'wb') as f:
        pickle.dump(bounds_df, f)

    print(f"Total patches : {len(coords_df):,}")
    print(f"Total scans   : {len(bounds_df)}")
    print(f"Sauvegardé dans {OUT_DIR}/")
    return coords_df, bounds_df

# Train (tous sauf TEST_FOLD)
train_uids = [uid for uid, info in UID_INDEX.items() if info['subset'] != TEST_FOLD]
print(f"Train scans : {len(train_uids)}")
coords_train, bounds_train = build_coords_bounds(train_uids, f'train_fold{TEST_FOLD}')

In [ ]:
# Test (subset TEST_FOLD uniquement)
test_uids = [uid for uid, info in UID_INDEX.items() if info['subset'] == TEST_FOLD]
print(f"Test scans : {len(test_uids)}")
coords_test, bounds_test = build_coords_bounds(test_uids, f'test_fold{TEST_FOLD}')

In [ ]:
import pickle
import pandas as pd

# Vérification coords + bounds
with open('/kaggle/working/luna16_coords/coords_train_fold0.pkl', 'rb') as f:
    coords_train = pickle.load(f)
with open('/kaggle/working/luna16_coords/bounds_train_fold0.pkl', 'rb') as f:
    bounds_train = pickle.load(f)
with open('/kaggle/working/luna16_coords/coords_test_fold0.pkl', 'rb') as f:
    coords_test = pickle.load(f)
with open('/kaggle/working/luna16_coords/bounds_test_fold0.pkl', 'rb') as f:
    bounds_test = pickle.load(f)

print("=== coords_train ===")
print(coords_train.shape)
print(coords_train.head())
print(coords_train.dtypes)

print("\n=== bounds_train ===")
print(bounds_train.shape)
print(bounds_train.head())

print("\n=== Vérification cohérence ===")
print(f"Dernier end_id bounds : {bounds_train['end_id'].max()}")
print(f"Dernier id coords     : {coords_train['id'].max()}")
print(f"Match                 : {bounds_train['end_id'].max() == coords_train['id'].max()}")

print(f"\nPatches/scan moyen train : {len(coords_train) / len(bounds_train):.0f}")
print(f"Patches/scan moyen test  : {len(coords_test)  / len(bounds_test):.0f}")

In [ ]:
# luna_dataset.py

import os
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
import SimpleITK as sitk
from torchvision import transforms

# UID_INDEX, load_ct_seg définis dans luna_utils (cellule précédente)

class PatchSampler(Sampler):
    """Même logique que CAMELYON : parcourt scan par scan."""

    FILL_TOKEN     = -1
    SLIDE_END_TOKEN = -2

    def __init__(self, bounds_df, batch_size=256):
        self.bounds_df  = bounds_df
        self.batch_size = batch_size
        self.num_scans  = len(bounds_df)

    def __len__(self):
        total = 0
        for _, row in self.bounds_df.iterrows():
            n = row['end_id'] - row['start_id'] + 1
            remainder = (n + 1) % self.batch_size
            pad = self.batch_size - remainder
            total += n + pad + 1  # +1 pour SLIDE_END_TOKEN
        return total

    def __iter__(self):
        all_idx = []
        for _, row in self.bounds_df.iterrows():
            start_id = row['start_id']
            end_id   = row['end_id']
            patch_idx = list(range(start_id, end_id + 1))
            n = len(patch_idx)

            # Padding pour compléter le batch
            remainder = (n + 1) % self.batch_size
            pad = self.batch_size - remainder
            patch_idx += [self.FILL_TOKEN] * pad

            # Token fin de scan
            patch_idx.append(self.SLIDE_END_TOKEN)
            all_idx.extend(patch_idx)
        return iter(all_idx)


class LunaImages(Dataset):
    """
    Charge les patches CT directement depuis les fichiers .mhd.
    Utilisé pour l'extraction de features après BYOL.
    """

    def __init__(self, uid_index, coords_df, patch_size=64):
        self.uid_index  = uid_index
        self.coords_df  = coords_df
        self.patch_size = patch_size

        self.transform = transforms.Compose([
            transforms.ToTensor(),                        # (1, H, W) float32
            transforms.CenterCrop(56),                    # légère crop centrale
            transforms.Resize(64),                        # retour 64×64
        ])

        self.current_uid  = None
        self.current_arr  = None

    def __len__(self):
        return len(self.coords_df)

    def _load_scan(self, uid):
        info = self.uid_index[uid]
        arr_norm, arr_seg, _, _ = load_ct_seg(info)
        return arr_norm  # (Z, Y, X) float32 [0,1]

    def __getitem__(self, i):
        data = {}
        is_empty = i < 0

        if not is_empty:
            row = self.coords_df.iloc[i]
            uid    = row['name']
            x, y, z = int(row['x']), int(row['y']), int(row['z'])
            pos_id = row['pos_id']

            # Charger le scan si changement
            if uid != self.current_uid:
                self.current_arr = self._load_scan(uid)
                self.current_uid = uid

            # Extraire patch 2D (z, y:y+ps, x:x+ps)
            ps = self.patch_size
            patch = self.current_arr[z, y:y+ps, x:x+ps]  # (H, W) float32

            # Ajouter dim canal → (1, H, W)
            patch_tensor = torch.from_numpy(patch).unsqueeze(0)

            label = self.uid_index[uid]['label']

            data['patch']      = patch_tensor
            data['label']      = label
            data['pos_id']     = pos_id
            data['slide_name'] = uid
        else:
            data['patch']      = torch.zeros(1, self.patch_size, self.patch_size)
            data['label']      = -1
            data['pos_id']     = 9999
            data['slide_name'] = ''

        data['data_id'] = i
        return data


class LunaFeatures(Dataset):
    """
    Charge les features pré-extraites depuis un fichier HDF5.
    Utilisé pour la classification avec IPS + attention.
    """

    def __init__(self, conf, train=True):
        import h5py
        self.tasks    = conf.tasks
        filename      = conf.train_fname if train else conf.test_fname
        self.data_dir = os.path.join(conf.data_dir, filename)
        self._select_scans()

    def _select_scans(self):
        import h5py
        with h5py.File(self.data_dir, 'r') as f:
            self.scan_names = list(f.keys())
        self.data_len = len(self.scan_names)

    def open_hdf5(self):
        import h5py
        self.dataset = h5py.File(self.data_dir, 'r')

    def __len__(self):
        return self.data_len

    def __getitem__(self, i):
        if not hasattr(self, 'dataset'):
            self.open_hdf5()

        scan_name = self.scan_names[i]
        scan      = self.dataset[scan_name]
        patches   = scan['img'][:]       # (N_patches, 2048)
        label     = scan.attrs['label']

        data_dict = {'input': patches}
        for task in self.tasks.values():
            data_dict[task['name']] = label
        return data_dict

In [ ]:
# byol_augmentations.py

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import copy
import random

# ── Augmentations BYOL adaptées CT grayscale ──
# Pas de jitter couleur ni niveaux de gris (déjà grayscale)
# Pas de solarisation (pas de sens en HU normalisé)
# On garde : flip, rotation, crop, flou gaussien

class BYOLAugment:
    """
    Génère 2 vues augmentées d'un patch CT grayscale (1, H, W).
    Inspiré du tableau 9 de l'article (adapté pour CT 1 canal).
    """
    def __init__(self, patch_size=64):
        self.view1 = T.Compose([
            T.RandomResizedCrop(patch_size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomApply([T.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))], p=0.5),
            T.RandomRotation(degrees=180),
        ])
        self.view2 = T.Compose([
            T.RandomResizedCrop(patch_size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomApply([T.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))], p=0.8),
            T.RandomRotation(degrees=180),
        ])

    def __call__(self, patch):
        """
        patch : torch.Tensor (1, H, W) float32 [0, 1]
        retourne : (view1, view2) chacune (1, H, W)
        """
        return self.view1(patch), self.view2(patch)


# ── Modèle BYOL corrigé ──
class MLP(nn.Module):
    def __init__(self, in_dim=2048, hidden_dim=4096, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)


class EncoderWithProjector(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet50(weights=None)
        # 1 canal grayscale
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2,
                                padding=3, bias=False)
        self.encoder   = nn.Sequential(*list(base.children())[:-1])  # (B,2048,1,1)
        self.projector = MLP(in_dim=2048, hidden_dim=4096, out_dim=256)

    def forward(self, x):
        h = self.encoder(x).flatten(start_dim=1)   # (B, 2048)
        z = self.projector(h)                        # (B, 256)
        return h, z


class BYOLModel(nn.Module):
    def __init__(self, tau=0.996):
        super().__init__()
        self.tau       = tau
        self.online    = EncoderWithProjector()
        self.predictor = MLP(in_dim=256, hidden_dim=4096, out_dim=256)
        self.target    = copy.deepcopy(self.online)
        for p in self.target.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update_target(self):
        for p_o, p_t in zip(self.online.parameters(),
                             self.target.parameters()):
            p_t.data = self.tau * p_t.data + (1 - self.tau) * p_o.data

    @staticmethod
    def loss_fn(p, z):
        p = nn.functional.normalize(p, dim=-1)
        z = nn.functional.normalize(z, dim=-1)
        return 2 - 2 * (p * z).sum(dim=-1).mean()

    def forward(self, x1, x2):
        _, z1_online = self.online(x1)
        _, z2_online = self.online(x2)
        p1 = self.predictor(z1_online)
        p2 = self.predictor(z2_online)
        with torch.no_grad():
            _, z1_target = self.target(x1)
            _, z2_target = self.target(x2)
        loss = (self.loss_fn(p1, z1_target) +
                self.loss_fn(p2, z2_target)) * 0.5
        return loss


# ── Test ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = BYOLModel().to(device)
aug    = BYOLAugment(patch_size=64)

# Simuler un batch de patches
patches = torch.rand(4, 1, 64, 64)
v1_list, v2_list = [], []
for p in patches:
    v1, v2 = aug(p)
    v1_list.append(v1)
    v2_list.append(v2)

x1 = torch.stack(v1_list).to(device)
x2 = torch.stack(v2_list).to(device)

loss = model(x1, x2)
model.update_target()

print(f"Device          : {device}")
print(f"x1 shape        : {x1.shape}")
print(f"Loss            : {loss.item():.4f}")
print(f"Params online   : {sum(p.numel() for p in model.online.parameters()):,}")
print(f"Augmentations   : OK")

In [ ]:
# byol_dataset.py

import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
import random

# UID_INDEX, load_ct_seg, BYOLAugment définis dans cellules précédentes

class BYOLPatchSampler(Sampler):
    """
    Tire aléatoirement N patches par époque depuis tous les scans.
    Comme CAMELYON : 270 000 patches/époque tirés au hasard.
    """
    def __init__(self, total_patches, n_samples_per_epoch):
        self.total_patches      = total_patches
        self.n_samples_per_epoch = n_samples_per_epoch

    def __len__(self):
        return self.n_samples_per_epoch

    def __iter__(self):
        idx = random.sample(range(self.total_patches), self.n_samples_per_epoch)
        return iter(idx)


# byol_dataset_v2.py — version avec pré-chargement RAM

import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
import random
import pickle
from tqdm import tqdm

class LunaBYOLDatasetFast(Dataset):
    """
    Pré-charge TOUS les patches en RAM au démarrage.
    32GB RAM dispo, ~4M patches × 64×64 × float16 = ~16GB → OK
    """
    def __init__(self, uid_index, coords_df, patch_size=64):
        self.patch_size = patch_size
        self.aug        = BYOLAugment(patch_size=patch_size)
        self.patches    = None  # sera rempli par preload()
        self._preload(uid_index, coords_df)

    def _preload(self, uid_index, coords_df):
        print("Pré-chargement des patches en RAM...")

        # Estimer la taille
        n       = len(coords_df)
        ps      = self.patch_size
        # float16 pour économiser : 4M × 64 × 64 × 2 bytes = ~32GB → trop
        # float16 : 4M × 64 × 64 × 2 = 32GB  ← trop juste
        # On stocke en uint8 [0,255] : 4M × 64 × 64 × 1 = 16GB ← OK
        self.patches = np.empty((n, ps, ps), dtype=np.uint8)

        current_uid = None
        current_arr = None

        for i, row in tqdm(coords_df.iterrows(), total=n, desc="Loading patches"):
            uid  = row['name']
            x, y, z = int(row['x']), int(row['y']), int(row['z'])

            if uid != current_uid:
                info = uid_index[uid]
                arr_norm, _, _, _ = load_ct_seg(info)
                # Convertir float32[0,1] → uint8[0,255] pour économiser RAM
                current_arr = (arr_norm * 255).astype(np.uint8)
                current_uid = uid

            self.patches[i] = current_arr[z, y:y+ps, x:x+ps]

        print(f"RAM patches : {self.patches.nbytes / 1e9:.1f} GB")
        print(f"Pré-chargement terminé : {n:,} patches")

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, i):
        # uint8 → float32 [0,1]
        patch = torch.from_numpy(
            self.patches[i].astype(np.float32) / 255.0
        ).unsqueeze(0)  # (1, H, W)
        v1, v2 = self.aug(patch)
        return v1, v2


# ── Test mémoire avant de tout charger ──
import psutil
ram_avail = psutil.virtual_memory().available / 1e9
n_patches = len(coords_train)
ps        = 64
ram_needed = n_patches * ps * ps * 1 / 1e9  # uint8

print(f"RAM disponible   : {ram_avail:.1f} GB")
print(f"RAM nécessaire   : {ram_needed:.1f} GB  (uint8)")
print(f"Margin           : {ram_avail - ram_needed:.1f} GB")
print(f"\nLancer le pré-chargement ? (oui si margin > 4GB)")

In [ ]:
import numpy as np
import os
import pickle
from tqdm import tqdm

CACHE_DIR   = '/kaggle/working/luna16_cache'
PATCH_SIZE  = 64
os.makedirs(CACHE_DIR, exist_ok=True)

patches_path = f'{CACHE_DIR}/patches_train_fold0.npy'
coords_path  = '/kaggle/working/luna16_coords/coords_train_fold0.pkl'

with open(coords_path, 'rb') as f:
    coords_train = pickle.load(f)

n  = len(coords_train)
ps = PATCH_SIZE

if not os.path.exists(patches_path):
    print(f"Création du cache disque : {n:,} patches → {patches_path}")
    # memmap en écriture — stocké sur disque, pas en RAM
    patches_mm = np.memmap(patches_path, dtype=np.uint8,
                           mode='w+', shape=(n, ps, ps))

    current_uid = None
    current_arr = None

    for i, row in tqdm(coords_train.iterrows(), total=n, desc="Caching"):
        uid     = row['name']
        x, y, z = int(row['x']), int(row['y']), int(row['z'])

        if uid != current_uid:
            info = UID_INDEX[uid]
            arr_norm, _, _, _ = load_ct_seg(info)
            current_arr = (arr_norm * 255).astype(np.uint8)
            current_uid = uid

        patches_mm[i] = current_arr[z, y:y+ps, x:x+ps]

    patches_mm.flush()
    print(f"Cache sauvegardé : {os.path.getsize(patches_path)/1e9:.1f} GB sur disque")

else:
    print(f"Cache déjà existant : {patches_path}")
    print(f"Taille              : {os.path.getsize(patches_path)/1e9:.1f} GB")

# Charger en memmap lecture — accès disque à la demande, ~0 RAM
patches_mm = np.memmap(patches_path, dtype=np.uint8,
                       mode='r', shape=(n, ps, ps))
print(f"Memmap chargé : shape={patches_mm.shape}  dtype={patches_mm.dtype}")
print(f"RAM utilisée  : quasi nulle (lecture disque à la demande)")

In [ ]:
# byol_dataset_memmap.py

import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
import random

class LunaBYOLDatasetMemmap(Dataset):
    """
    Dataset BYOL utilisant le cache memmap (quasi 0 RAM).
    patches_mm : np.memmap shape (N, 64, 64) dtype uint8
    Lecture disque à la demande → pas de pré-chargement RAM.
    """
    def __init__(self, patches_mm, patch_size=64):
        self.patches    = patches_mm          # np.memmap mode='r'
        self.patch_size = patch_size
        self.aug        = BYOLAugment(patch_size=patch_size)

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, i):
        # Copie explicite pour éviter warning memmap → numpy
        patch_np = self.patches[i].copy()                      # (64, 64) uint8

        # uint8 [0,255] → float32 [0,1] → tensor (1, H, W)
        patch = torch.from_numpy(
            patch_np.astype(np.float32) / 255.0
        ).unsqueeze(0)

        v1, v2 = self.aug(patch)
        return v1, v2


class BYOLRandomSampler(Sampler):
    """
    Tire n_samples indices aléatoires sans remise par époque.
    Remplace le shuffle standard pour contrôler exactement
    le nombre de patches vus par époque (270 000).
    """
    def __init__(self, total_patches, n_samples_per_epoch):
        self.total_patches       = total_patches
        self.n_samples_per_epoch = n_samples_per_epoch

    def __len__(self):
        return self.n_samples_per_epoch

    def __iter__(self):
        idx = random.sample(range(self.total_patches), self.n_samples_per_epoch)
        return iter(idx)


# ── Instanciation ──
N_SAMPLES_EPOCH = 270_000
BATCH_SIZE      = 256

dataset = LunaBYOLDatasetMemmap(patches_mm, patch_size=64)

sampler = BYOLRandomSampler(
    total_patches       = len(dataset),
    n_samples_per_epoch = N_SAMPLES_EPOCH,
)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = 4,
    pin_memory  = True,
    drop_last   = True,
    persistent_workers = True,   # évite de re-fork les workers à chaque époque
)

# ── Vérification ──
v1, v2 = next(iter(loader))

print(f"Dataset size     : {len(dataset):,} patches")
print(f"Patches/époque   : {N_SAMPLES_EPOCH:,}")
print(f"Steps/époque     : {N_SAMPLES_EPOCH // BATCH_SIZE}")
print(f"v1 shape         : {v1.shape}")   # (256, 1, 64, 64)
print(f"v2 shape         : {v2.shape}")
print(f"v1 dtype         : {v1.dtype}")
print(f"v1 min/max       : {v1.min():.3f} / {v1.max():.3f}")
print(f"DataLoader       : OK")

In [ ]:
# byol_train_v2.py

import torch
import torch.nn as nn
import math
import os
import time
from torch.amp import GradScaler, autocast

# ── Hyperparamètres ──
N_EPOCHS        = 500
BATCH_SIZE      = 256
LR              = 3e-4
WD              = 1.5e-6
TAU_BASE        = 0.996
WARMUP_EPOCHS   = 10
CHECKPOINT_DIR  = '/kaggle/working/byol_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Modèle ──
model = BYOLModel(tau=TAU_BASE).to(device)

# ── Optimiseur ──
optimizer = torch.optim.AdamW(
    list(model.online.parameters()) + list(model.predictor.parameters()),
    lr=LR, weight_decay=WD
)

# ── Scheduler cosine avec warmup ──
def get_lr(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS          # warmup linéaire
    progress = (epoch - WARMUP_EPOCHS) / (N_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress)) # cosine decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

# ── Mixed precision ──
scaler = GradScaler('cuda')

# ── EMA tau schedule (augmente de TAU_BASE → 1 au fil des epochs) ──
def get_tau(epoch):
    progress = epoch / N_EPOCHS
    return 1 - (1 - TAU_BASE) * (math.cos(math.pi * progress) + 1) / 2

# ── Reprise depuis checkpoint si existant ──
start_epoch = 0
best_loss   = float('inf')

ckpt_path = os.path.join(CHECKPOINT_DIR, 'byol_best.pth')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optim'])
    start_epoch = ckpt['epoch'] + 1
    best_loss   = ckpt['loss']
    print(f"Reprise depuis epoch {start_epoch}  (best_loss={best_loss:.4f})")
else:
    print("Entraînement from scratch")

# ── Boucle principale ──
steps_per_epoch = N_SAMPLES_EPOCH // BATCH_SIZE   # 1054

for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    model.tau = get_tau(epoch)

    epoch_loss = 0.0
    t0         = time.time()

    for step, (v1, v2) in enumerate(loader):
        v1 = v1.to(device, non_blocking=True)
        v2 = v2.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda'):
            loss = model(v1, v2)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        model.update_target()

        epoch_loss += loss.item()

    scheduler.step()

    avg_loss = epoch_loss / steps_per_epoch
    elapsed  = time.time() - t0
    lr_now   = optimizer.param_groups[0]['lr']
    tau_now  = model.tau

    print(f"Epoch {epoch+1:03d}/{N_EPOCHS} | "
          f"loss={avg_loss:.4f} | "
          f"lr={lr_now:.2e} | "
          f"tau={tau_now:.4f} | "
          f"time={elapsed:.0f}s")

    # ── Checkpoints ──
    state = {
        'epoch': epoch,
        'loss' : avg_loss,
        'model': model.state_dict(),
        'optim': optimizer.state_dict(),
    }

    # Meilleur modèle
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(state, os.path.join(CHECKPOINT_DIR, 'byol_best.pth'))

    # Toutes les 50 epochs
    if (epoch + 1) % 50 == 0:
        torch.save(state, os.path.join(CHECKPOINT_DIR, f'byol_epoch{epoch+1}.pth'))

print(f"\nEntraînement terminé. Best loss : {best_loss:.4f}")
print(f"Checkpoint : {CHECKPOINT_DIR}/byol_best.pth")

In [ ]:
# extract_features.py

import torch
import torch.nn as nn
import numpy as np
import h5py
import pickle
import os
from tqdm import tqdm

# ── Config ──
CHECKPOINT_PATH = '/kaggle/working/byol_checkpoints/byol_best.pth'
HDF5_DIR        = '/kaggle/working/luna16_features'
BATCH_SIZE      = 512   # plus grand qu'en train, pas de backprop
PATCH_SIZE      = 64
os.makedirs(HDF5_DIR, exist_ok=True)

# ── Charger l'encodeur depuis le checkpoint ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt  = torch.load(CHECKPOINT_PATH, map_location=device)
model = BYOLModel(tau=0.996).to(device)
model.load_state_dict(ckpt['model'])
print(f"Checkpoint chargé — epoch {ckpt['epoch']+1}, loss={ckpt['loss']:.4f}")

# Garder uniquement l'encodeur, gelé
encoder = model.online.encoder.to(device)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

print(f"Encodeur : {sum(p.numel() for p in encoder.parameters()):,} params (gelé)")


# ── Fonction d'extraction pour un split ──
def extract_split(coords_path, bounds_path, cache_path, out_fname, uid_index, split_name):
    print(f"\n{'='*50}")
    print(f"Extraction : {split_name}")

    with open(coords_path, 'rb') as f:
        coords_df = pickle.load(f)
    with open(bounds_path, 'rb') as f:
        bounds_df = pickle.load(f)

    n_total = len(coords_df)
    ps      = PATCH_SIZE

    # Charger le cache memmap
    patches_mm = np.memmap(cache_path, dtype=np.uint8,
                           mode='r', shape=(n_total, ps, ps))

    out_path = os.path.join(HDF5_DIR, out_fname)

    with h5py.File(out_path, 'w') as hf:
        for _, row in tqdm(bounds_df.iterrows(), total=len(bounds_df),
                           desc=f"Scans {split_name}"):
            scan_name = row['names']
            start_id  = int(row['start_id'])
            end_id    = int(row['end_id'])
            n_patches = end_id - start_id + 1

            label = uid_index[scan_name]['label']

            # Extraire les features par batch
            all_features = []

            for batch_start in range(start_id, end_id + 1, BATCH_SIZE):
                batch_end = min(batch_start + BATCH_SIZE, end_id + 1)

                # uint8 → float32 [0,1] → tensor (B, 1, H, W)
                batch_np = patches_mm[batch_start:batch_end].copy()
                batch    = torch.from_numpy(
                    batch_np.astype(np.float32) / 255.0
                ).unsqueeze(1).to(device, non_blocking=True)  # (B, 1, 64, 64)

                with torch.no_grad():
                    feats = encoder(batch).flatten(start_dim=1)  # (B, 2048)

                all_features.append(feats.cpu().numpy())

            features = np.concatenate(all_features, axis=0)  # (N_patches, 2048)

            # Sauvegarder dans HDF5
            grp = hf.create_group(scan_name)
            grp.create_dataset('img', data=features, dtype=np.float32)
            grp.attrs['label'] = label

    print(f"Sauvegardé : {out_path}")
    print(f"Taille     : {os.path.getsize(out_path)/1e9:.2f} GB")
    return out_path


# ── Extraction train ──
train_path = extract_split(
    coords_path = '/kaggle/working/luna16_coords/coords_train_fold0.pkl',
    bounds_path = '/kaggle/working/luna16_coords/bounds_train_fold0.pkl',
    cache_path  = '/kaggle/working/luna16_cache/patches_train_fold0.npy',
    out_fname   = 'features_train_fold0.h5',
    uid_index   = UID_INDEX,
    split_name  = 'TRAIN',
)

# ── Extraction test ──
# Cache test à créer si pas encore fait
TEST_CACHE = '/kaggle/working/luna16_cache/patches_test_fold0.npy'

if not os.path.exists(TEST_CACHE):
    print("\nCréation cache test...")
    coords_test_path = '/kaggle/working/luna16_coords/coords_test_fold0.pkl'
    with open(coords_test_path, 'rb') as f:
        coords_test = pickle.load(f)

    n_test = len(coords_test)
    patches_test_mm = np.memmap(TEST_CACHE, dtype=np.uint8,
                                mode='w+', shape=(n_test, PATCH_SIZE, PATCH_SIZE))

    current_uid = None
    current_arr = None
    from tqdm import tqdm

    for i, row in tqdm(coords_test.iterrows(), total=n_test, desc="Cache test"):
        uid     = row['name']
        x, y, z = int(row['x']), int(row['y']), int(row['z'])
        if uid != current_uid:
            info = UID_INDEX[uid]
            arr_norm, _, _, _ = load_ct_seg(info)
            current_arr = (arr_norm * 255).astype(np.uint8)
            current_uid = uid
        patches_test_mm[i] = current_arr[z, y:y+PATCH_SIZE, x:x+PATCH_SIZE]

    patches_test_mm.flush()
    print(f"Cache test créé : {os.path.getsize(TEST_CACHE)/1e9:.1f} GB")

test_path = extract_split(
    coords_path = '/kaggle/working/luna16_coords/coords_test_fold0.pkl',
    bounds_path = '/kaggle/working/luna16_coords/bounds_test_fold0.pkl',
    cache_path  = TEST_CACHE,
    out_fname   = 'features_test_fold0.h5',
    uid_index   = UID_INDEX,
    split_name  = 'TEST',
)

# ── Vérification HDF5 ──
print("\n── Vérification ──")
with h5py.File(train_path, 'r') as hf:
    keys   = list(hf.keys())
    sample = hf[keys[0]]
    print(f"Train scans      : {len(keys)}")
    print(f"Exemple scan     : {keys[0]}")
    print(f"Features shape   : {sample['img'].shape}")   # (N_patches, 2048)
    print(f"Label            : {sample.attrs['label']}")

with h5py.File(test_path, 'r') as hf:
    print(f"Test scans       : {len(hf.keys())}")